# Policies Analysis Using SDK

This notebook demonstrates how analysts can use the PoliciesAnalyst SDK to analyze stop loss insurance policies data.

## Overview

**Important**: Analysts should ONLY use SDK methods. Do NOT access database or files directly.

The SDK provides high-level, analyst-friendly methods for:
- Loading policies data from database (via SDK)
- Filtering active policies
- Finding policies by employer
- Calculating total coverage
- Converting to pandas DataFrames for analysis

All business logic is encapsulated in the SDK.

## Setup

First, install the package in editable mode:
```bash
pip install -e .
```


In [ ]:
from src.sdk import PoliciesAnalyst

# Initialize the analyst
# SDK handles database connection internally
analyst = PoliciesAnalyst(db_path="../warehouse.db")

print("PoliciesAnalyst SDK initialized")
print("Ready to load data from database")


## Load Policies Data

Load policies from the database. The SDK handles all database access.

**Note**: Analysts should NOT access the database directly.


In [ ]:
# Load policies from database (silver layer - processed data)
policies = analyst.load_from_database(layer="silver")

print(f"Loaded {len(policies)} policies from database")
print(f"\nFirst policy example:")
if policies:
    first_policy = policies[0]
    print(f"  Policy ID: {first_policy.policy_id}")
    print(f"  Employer ID: {first_policy.employer_id}")
    print(f"  Stop Loss Limit: ${first_policy.stop_loss_limit:,.2f}")
    print(f"  Status: {first_policy.status}")
    print(f"  Active: {first_policy.is_active()}")


## Get Summary Statistics

Get pre-calculated summary statistics from SDK.


In [ ]:
# Get summary statistics - SDK does all calculations
stats = analyst.get_summary_statistics(policies)

print("Summary Statistics:")
print(f"  Total Policies: {stats['total_policies']}")
print(f"  Total Coverage: ${stats['total_coverage']:,.2f}")
print(f"  Average Coverage: ${stats['average_coverage']:,.2f}")
print(f"\nBy Status:")
for status, count in stats['by_status'].items():
    print(f"  {status}: {count}")
print(f"\nBy Employer:")
for employer, count in list(stats['by_employer'].items())[:5]:  # Show first 5
    print(f"  {employer}: {count}")


## Find Policies by Employer

Get all policies for a specific employer.


In [ ]:
# Filter active policies - SDK handles the logic
active = analyst.get_active_policies(policies)
print(f"Active policies: {len(active)}")

if active:
    active_coverage = analyst.get_total_coverage(active)
    print(f"Total active coverage: ${active_coverage:,.2f}")


In [ ]:
# Get policies for a specific employer
employer_id = "EMP-ACME-CORP"
employer_policies = analyst.get_employer_policies(policies, employer_id)

print(f"Policies for {employer_id}: {len(employer_policies)}")

if employer_policies:
    employer_coverage = analyst.get_total_coverage(employer_policies)
    print(f"Total coverage: ${employer_coverage:,.2f}")
    
    for policy in employer_policies:
        print(f"  {policy.policy_id}: ${policy.stop_loss_limit:,.2f}")


## Calculate Total Coverage

Calculate the total stop loss coverage across all policies.


In [ ]:
# Calculate total coverage
total_coverage = analyst.get_total_coverage(policies)
print(f"Total stop loss coverage: ${total_coverage:,.2f}")
print(f"Average coverage per policy: ${total_coverage / len(policies):,.2f}")


## Convert to Pandas DataFrame

Convert policies to a pandas DataFrame for advanced analysis.


In [ ]:
# Convert to DataFrame
df = analyst.to_dataframe(policies)

print(f"DataFrame shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print("\nFirst few rows:")
df.head()


## DataFrame Analysis

Use pandas for advanced analysis.


In [ ]:
# Summary statistics
print("Stop Loss Limit Statistics:")
print(df['stop_loss_limit'].describe())

print("\n\nStatus Distribution:")
print(df['status'].value_counts())

print("\n\nPolicies by Employer:")
print(df['employer_id'].value_counts())
